In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.types import BOOLEAN, BIGINT, VARCHAR, TEXT, INTEGER, DOUBLE_PRECISION
from sqlalchemy.dialects.postgresql.types import TIMESTAMP

from telegram_quality_control.db import get_conn_string

import polars as pl

from fastparquet import write
from tqdm import tqdm

In [ ]:
scratch_folder = Path(os.getenv("SCRATCH_FOLDER")) / "data_export"
for file in scratch_folder.glob("*"):
    print(file)

In [ ]:
db_url = get_conn_string()

engine = create_engine(db_url)

tables = {
    "chat_language": "chat_id",
    "chats": "id",
    "chats_users": None,
    "entity_hashtags": "entity_id",
    "entity_urls": "entity_id",
    "messages": "id",
    "poll_options": "id",
    "polls": "id",
    "reactions": "id",
    "users": "id",
}

# convert datatypes from sqlalchemy to polars
type_map = {
    BOOLEAN: pl.Boolean,
    BIGINT: pl.Int64,
    VARCHAR: pl.String,
    DOUBLE_PRECISION: pl.Float64,
    TEXT: pl.String,
    INTEGER: pl.Int64,
    TIMESTAMP: pl.Datetime("us"),
}

In [ ]:
conn = engine.connect()

batch_size = 100_000_000

inspector = inspect(engine)

drop_columns = ["crawl_created_at"]

for table_name, index in tables.items():
    print(f"Processing table {table_name}...")

    num_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
    if num_rows > batch_size:
        num_batches = (num_rows + batch_size - 1) // batch_size
    else:
        num_batches = 1

    if num_batches == 1:
        save_path = scratch_folder / f"{table_name}.parquet"
        if save_path.exists():
            continue
    else:
        save_path = scratch_folder / table_name
        # if save_path.exists():
        #     continue
        save_path.mkdir(parents=True, exist_ok=True)

    print(f"Exporting {table_name} in {num_batches} batches...")
    columns = inspector.get_columns(table_name)
    schema = {}
    for col in columns:
        col_type = col["type"].__class__
        pl_type = type_map[col_type]
        schema[col["name"]] = pl_type

    print(schema)

    for i, df in tqdm(
        enumerate(
            pl.read_database(
                query=f"SELECT * FROM {table_name}",
                connection=conn.execution_options(stream_results=True),
                iter_batches=True,
                batch_size=batch_size,
                schema_overrides=schema,
            )
        ),
        total=num_batches,
    ):
        for col in drop_columns:
            if col in df.columns:
                df = df.drop(col)

        if num_batches == 1:
            df.write_parquet(save_path, compression="zstd")
        else:
            if (save_path / f"{table_name}_batch_{i}.parquet").exists():
                continue
            df.write_parquet(save_path / f"{table_name}_batch_{i}.parquet", compression="zstd")

    print(f"Exported {table_name} to {save_path}")